# Unsupervised Text Classification

Using LLMs to categorize responses seems to be pretty unreliable and I have been unsuccessful in getting the models to cleanly work. 
In this document, I discuss using `BERTopic` to create categories of responses. This seems to be a fairly popular method in recent years.



## Questions

All responses can be found in `data/student-responses.csv`. In this file, all collected responses are stored from the bar chart and heatmap experiments. In the data cleaning script, a flag for bar chart or heatmap experiment was included.

In [7]:
# Import modules
import pandas as pd

# Read dataset
df = pd.read_csv("../data/student-responses.csv")

# Filter only Bar Chart experiment responses
df = df[df["experiment"] == "Bar chart"]
df.head()

,id,section_sis_id,section,attempt,What components of the experiment are clearer now than they were as a participant What questions do you still have for the experimenter Write 3 5 sentences reflecting on the abstract,Paste the response code you received after participating in the graphics experiment here,As of today I am at least 19 years of age,My instructor may share my reflection responses with the researchers in this study,What do you think the purpose of the experiment was,What elements of experimental design such as randomization or the use of a control group do you think were present in the experiment Why,...,What sources of error are involved in this experiment,What variables were examined For each variable identify whether it was quantitative or categorical,In this class you ll be learning about the process of scientific investigation What do you think that process looks like from the perspective of a researcher compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results Write a paragraph at least 3 5 sentences about how you think science happens,How did the information you gained from the components of this project participation post study reflection extended abstract presentation differ,If you had to hear about this study using only the extended abstract or only the presentation which one would you prefer Which one would be better for determining whether the experiment was well designed,What components were emphasized in the presentation that weren t emphasized in the abstract Why do you think that is,What critiques do you have of this study and its design What would have made the study better,year,semester,experiment
0,15464f12481fd2057f7a8205a42e03e8,STAT-218-011.1241,1,1,Seeing the marked bar closer and in person rat...,40646-92578-95520-71653,True,I agree,To see how accurate the students can be with e...,I think this was a paried control group becaus...,...,There's no results in whether if you were clos...,Circle and Triangle graph. Quantitative,I think that’ll look a lot different to a pers...,They gave us charts and information that shows...,"Extended abstract, extended abstarct",2d and 3d bar graphs because they were trying ...,How do you know whether or not your response i...,2024,Spring,Bar chart
1,2800ab155946ad01714a453f448edbae,STAT-218-011.1241,1,1,I think it is clearer that this experiment is ...,53670-79351-03305-70518,True,I agree,I think the purpose was to determine is we cou...,I think the control group was the models we ha...,...,"It might be a sampling error, also people may ...",Model size- quantitative color- categorical,A researcher has to form a hypothesis and crea...,I think the presentation was the easiest to un...,I would prefer to hear about the study from th...,I think the presentation emphasized more of th...,I think this study has a big possibility for c...,2024,Spring,Bar chart
2,43c3004415a1e8390ac4e7df8d06a292,STAT-218-011.1241,1,1,The whole objective of the experiment is more ...,13531-97095-57767-17258,True,I agree,The purpose of the experiment was to determine...,Randomization was used because we were given d...,...,The people who did the experiment could have m...,The height which was quantitative and the othe...,The process of scientific investigation is jus...,During the participation and post-study reflec...,I would prefer the video just because abstract...,The component that was emphasized to me was th...,Maybe to explain the experiment better to the ...,2024,Spring,Bar chart
3,629669a5f6966aac68902888966f15be,STAT-218-011.1241,1,1,NaN,12068-97156-84567-38829,True,I agree,NaN,NaN,...,NaN,NaN,I believe that it will consist of the person b...,NaN,NaN,NaN,NaN,2024,Spring,Bar chart
4,7b3570d08b16b612f899afa0a3bc4801,NaN,1,1,NaN,NaN,True,I agree,NaN,NaN,...,NaN,NaN,I believe that scientific investigation is the...,NaN,NaN,NaN,NaN,2024,Spring,Bar chart


In [8]:
df["section"].value_counts().to_frame(name="count").reset_index().rename(columns={"index": "section"}).sort_values("section")

,section,count
29,1,9
27,2,10
1,3,47
25,4,13
28,5,10
20,6,16
21,7,15
11,8,20
4,9,38
8,10,23


Next, it is worth noting that the quesitons are not in order. Below is a table summary of the questions and their corresponding modules.

| Module | Question Number | Dataframe Column Index (0-index) | Prompt |
|---|---:|---:|---|
| pre-experiment | Q1 | 13 | In this class, you’ll be learning about the process of scientific investigation. What do you think that process looks like, from the perspective of a researcher, compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results? Write a paragraph (at least 3-5 sentences) about how you think science happens. |
| post-experiment | Q2 | 8 | What do you think the purpose of the experiment was? |
| post-experiment | Q3 | 10 | What hypotheses might the experimenter have been testing? |
| post-experiment | Q4 | 11 | What sources of error are involved in this experiment? |
| post-experiment | Q5 | 12 | What variables were examined? For each variable, identify whether it was quantitative or categorical. |
| post-experiment | Q6 | 9 | What elements of experimental design, such as randomization or the use of a control group, do you think were present in the experiment? Why? |
| abstract reflection | Q7 | 4 | What components of the experiment are clearer now than they were as a participant? What questions do you still have for the experimenter? Write 3-5 sentences reflecting on the abstract. |
| presentation reflection | Q8 | 14 | How did the information you gained from the components of this project (participation, post-study reflection, extended abstract, presentation) differ? |
| presentation reflection | Q9 | 16 | What components were emphasized in the presentation that weren’t emphasized in the abstract? Why do you think that is? |
| presentation reflection | Q10 | 17 | What critiques do you have of this study and its design? What would have made the study better? |
| presentation reflection | Q11 | 15 | If you had to hear about this study using only the extended abstract or only the presentation, which one would you prefer? Which one would be better for determining whether the experiment was well designed? |

## BERTopic

BERTopic is an unsupervised text classification method that leverages clustering and dimensional reduction. 

**Webpage:** <https://maartengr.github.io/BERTopic/index.html>

1. 

**Guide:** <https://www.youtube.com/watch?v=v3SePt3fr9g>

In [9]:
# Modules for topic modeling
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import hdbscan
from umap import UMAP
import openai
from bertopic.representation import OpenAI

# UMAP parameters and seed for reproducibility (I think these are the defaults, but I mostly wanted to set the random state)
umap_model = UMAP(n_neighbors=15, 
                  n_components=5, 
                  min_dist=0.0, 
                  metric='cosine', 
                  random_state=42)


# Configure HDBSCAN to have more clusters
hdb = hdbscan.HDBSCAN(
    min_cluster_size=10,     # ↓ smaller = more clusters
    min_samples=2,          # ↓ more sensitive
    prediction_data=True,   # required for some BERTopic visualizations
    cluster_selection_epsilon=0.1  # encourages splitting
)

# Configure OpenAI for topic representation
client = openai.OpenAI(
    base_url="http://localhost:11434/v1",
    api_key='ollama'
)

representation_model = OpenAI(client, model='mistral')

# Fit model for Q1 responses
q1 = df.iloc[:, 15].dropna().astype(str).tolist()
topic_model = BERTopic(
    embedding_model="all-MiniLM-L6-v2", 
    umap_model=umap_model,
    n_gram_range = (1,5), # I doubt there will be n-grams with 5 words, but it may be helpful
    calculate_probabilities=True,
    hdbscan_model=hdb,
    verbose=True,
    representation_model=representation_model
)
topics, probs = topic_model.fit_transform(q1)


2026-04-06 16:43:02,754 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12307.37it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 13/13 [00:01<00:00,  7.10it/s]
2026-04-06 16:43:06,222 - BERTopic - Embedding - Completed ✓
2026-04-06 16:43:06,222 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 16:43:06,689 - BERTopic - Dimensionality - Completed ✓
2026-04-06 16:43:06,689 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 16:43:06,701 - BERTopic - Cluster - Completed ✓
2026-04-06 16:43:06,703 - BERTopic - Representation - Fine-tuning topics using representation m

In [10]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,25,-1_Preference for visual presentations,[Preference for visual presentations],"[I preferred the presentation more, it was eas..."
1,0,115,0_Study evaluation preferences,[Study evaluation preferences],[I would prefer the presentation only. Althoug...
2,1,55,1_Visual aids for experimental evaluation,[Visual aids for experimental evaluation],[I think I would prefer the presentation becau...
3,2,52,2_Preference for presentation design evaluation,[Preference for presentation design evaluation],"[I would prefer the presentation, because it s..."
4,3,44,3_Learning Preference for Presentations,[Learning Preference for Presentations],"[Personally, I would watch the presentation. T..."
5,4,35,4_Topic: Preference for study presentations vs...,[Topic: Preference for study presentations vs ...,[I would prefer the presentation. The presenta...
6,5,33,5_Preference for presentation over abstract (e...,[Preference for presentation over abstract (ex...,[The presentation is way more preffered to hea...
7,6,19,6_Value of Study Abstracts,[Value of Study Abstracts],[I prefer the abstract because it provides a m...
8,7,17,7_topics: Preferences for learning styles (stu...,[topics: Preferences for learning styles (stud...,[-Personally I am more able to understand mate...
9,8,10,8_Study Presentation Preference,[Study Presentation Preference],[I would prefer the presentation because it go...


In [11]:
# Perform hierarchical topic reduction
hierarchical_topics = topic_model.hierarchical_topics(q1)

# Visualize the subtopics
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 8/8 [00:04<00:00,  1.65it/s]


In [12]:
topic_model.visualize_topics()

In [13]:
topic_model.visualize_documents(q1)

## Function that fits model for specified question

In [14]:
def fit_bertopic(df, column_index):
    """
    Fit BERTopic on one dataframe column.

    Returns:
        topic_model: fitted BERTopic model
        topics: topic assignment list for each document
        probs: topic probability matrix from BERTopic
        docs: list of documents used for fitting
    """
    # Pull selected column, keeping only non-empty responses
    prompt_label = str(df.columns[column_index])
    responses = df.iloc[:, column_index]
    valid_mask = responses.notna() & responses.astype(str).str.strip().ne("")
    docs = responses[valid_mask].astype(str).tolist()

    if len(docs) == 0:
        raise ValueError(f"No non-empty responses found in column index {column_index}.")

    print(f"Running BERTopic for prompt: {prompt_label}")

    # UMAP setup
    umap_model = UMAP(
        n_neighbors=10,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    )

    # HDBSCAN setup
    hdb = hdbscan.HDBSCAN(
        min_cluster_size=5,
        min_samples=2,
        prediction_data=True,
        cluster_selection_epsilon=0.1
    )

    # OpenAI/Ollama-backed representation model setup
    client = openai.OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama"
    )
    representation_model = OpenAI(client, model="mistral")

    # Fit BERTopic
    topic_model = BERTopic(
        embedding_model="all-MiniLM-L6-v2",
        umap_model=umap_model,
        n_gram_range=(1, 5),
        calculate_probabilities=True,
        hdbscan_model=hdb,
        verbose=True,
        representation_model=representation_model
    )
    topics, probs = topic_model.fit_transform(docs)

    return topic_model, topics, probs, docs

In [15]:
# Example for Q1 responses (column index 13)
q1_topic_model, q1_topics, q1_probs, q1_docs = fit_bertopic(
    df=df,
    column_index=13
)

Running BERTopic for prompt: In this class you ll be learning about the process of scientific investigation What do you think that process looks like from the perspective of a researcher compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results Write a paragraph at least 3 5 sentences about how you think science happens


2026-04-06 16:43:24,376 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9835.47it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 20/20 [00:05<00:00,  3.87it/s]
2026-04-06 16:43:31,221 - BERTopic - Embedding - Completed ✓
2026-04-06 16:43:31,222 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 16:43:31,987 - BERTopic - Dimensionality - Completed ✓
2026-04-06 16:43:31,988 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 16:43:32,025 - BERTopic - Cluster - Completed ✓
2026-04-06 16:43:32,027 - BERTopic - Representation - Fine-tuning topics using representation mo

In [16]:
q1_topic_model.visualize_topics()

In [17]:
q1_hierarchical_topics = q1_topic_model.hierarchical_topics(df.iloc[:, 13].dropna().astype(str).tolist())
q1_topic_model.visualize_hierarchy(hierarchical_topics=q1_hierarchical_topics)

100%|██████████| 38/38 [00:42<00:00,  1.13s/it]


# Paper Draft

## Methods

**NOTE:** This section is only for these models. I have more in the actual manuscript.

Our sample is composed of students enrolled in STAT 218 at the University of Nebraska--Lincoln. While all students were required to participant in the experiential learning project as part of the course cirriculum, data was collected if students meet the age of majority in Nebraska (age 19 or older) and if they consented to data collection. The data collection took place between Summer 2023 and Spring 2025, where XX sections of STAT 218 participated. 


### Text Classification

In recent years, there has been a growing research area in Large-Language Models (LLMs) for classifying open-ended survey responses. Despite the promising aspect of using LLMs for classification, these models are sensitive to prompt and token limits (XXX), which can greatly influence the outputs. Additionally, these models tend to suffer from hallucinations and/or low accuracy rates compared to traditional human codings (XXX). The current capacity of LLMs are not yet ready for standalone classification, but they do have some integrations with non-zero box methods. 

For the classiciation of our responses, we focus on BERTopic (XXX), which is a topic modeling algorithm that incorporates clustering and dimensional reduction. BERTopic is highly customizable in each stage of the algorithm, allowing for multiple specifications for fine-tuning. Descriptions of this process can be found in Table (XXX), along with our specified models and hyper-parameters for each stage. We note a few of our chosen hyperparameter selections. For the clustering stage with HBDSCAN, we set the minimum cluster size to 5 so that smaller clusters can form. We found that the default settings were too restrictive and formed topics that closely aligned with the original prompt. We also included n-grams up to five words to allow for common phrases that students may have responded with (e.g., "scientific process"). Lastly, we incorporated Mistral (XXX) as a locally-run LLM to fine-tune the generated topic lists into interpretable categories, while also respecting concerns over data privacy of cloud-based LLMs.




| Stage | Step | Purpose | Option Used |
|---|---|---|---|
| Stage 1 | extract embeddings | Convert each response into a numeric vector that captures semantic meaning. | `embedding_model="all-MiniLM-L6-v2"` |
| Stage 2 | reduce dimensionality | Compress embeddings into a lower-dimensional space for better clustering efficiency | `UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)` |
| Stage 3 | cluster reduced embeddings | Group similar responses into topic clusters. | `HDBSCAN(min_cluster_size=5, min_samples=2, cluster_selection_epsilon=0.1, prediction_data=True)` |
| Stage 4 | tokenize topics | Break text into candidate terms/phrases used to represent each cluster. | `n_gram_range=(1, 5)` |
| Stage 5 | extract topic words | Compute the most representative words/phrases for each cluster. | BERTopic default c-TF-IDF weighting |
| Stage 6 | fine-tune topic representations | Improve readability and specificity of topic labels/keywords. | `representation_model=OpenAI(client, model="mistral")` with Ollama endpoint `http://localhost:11434/v1` |

## Results

### Pre-Experiment

The pre-experiment prompt consisted of asking participants to write a paragraph about how science differs from researchers and the general public. XXX students provided a response to this prompt.

> In this class, you’ll be learning about the process of scientific investigation. What do you think that process looks like, from the perspective of a researcher, compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results? Write a paragraph (at least 3-5 sentences) about how you think science happens.


In [18]:
# Q1
q1_topic_model, q1_topics, q1_probs, q1_docs = fit_bertopic(
    df=df,
    column_index=13
)

Running BERTopic for prompt: In this class you ll be learning about the process of scientific investigation What do you think that process looks like from the perspective of a researcher compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results Write a paragraph at least 3 5 sentences about how you think science happens


2026-04-06 16:44:58,451 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8869.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 20/20 [00:05<00:00,  3.73it/s]
2026-04-06 16:45:05,501 - BERTopic - Embedding - Completed ✓
2026-04-06 16:45:05,502 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 16:45:06,294 - BERTopic - Dimensionality - Completed ✓
2026-04-06 16:45:06,295 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 16:45:06,346 - BERTopic - Cluster - Completed ✓
2026-04-06 16:45:06,348 - BERTopic - Representation - Fine-tuning topics using representation mo

In [19]:
q1_topic_model.visualize_topics()

In [20]:
q1_hierarchical_topics = q1_topic_model.hierarchical_topics(q1_docs)
q1_topic_model.visualize_hierarchy(hierarchical_topics=q1_hierarchical_topics)

100%|██████████| 38/38 [00:43<00:00,  1.16s/it]


### Post-Experiment

#### Q2

**Prompt:**
> What do you think the purpose of the experiment was?

In [21]:
# Q2
q2_topic_model, q2_topics, q2_probs, q2_docs = fit_bertopic(
    df=df,
    column_index=8
)

2026-04-06 16:46:34,509 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic for prompt: What do you think the purpose of the experiment was


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13957.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:01<00:00, 16.10it/s]
2026-04-06 16:46:37,177 - BERTopic - Embedding - Completed ✓
2026-04-06 16:46:37,177 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 16:46:37,729 - BERTopic - Dimensionality - Completed ✓
2026-04-06 16:46:37,730 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 16:46:37,756 - BERTopic - Cluster - Completed ✓
2026-04-06 16:46:37,758 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 34/34 [00:24<00:00,  1.37it/s]
2026-04-06 16:47:02,751 - BERTop

In [22]:
q2_topic_model.visualize_topics()

In [23]:
q2_hierarchical_topics = q2_topic_model.hierarchical_topics(q2_docs)
q2_topic_model.visualize_hierarchy(hierarchical_topics=q2_hierarchical_topics)

100%|██████████| 32/32 [00:21<00:00,  1.50it/s]


#### Q3

**Prompt:**
> What hypotheses might the experimenter have been testing?

In [24]:
# Q3
q3_topic_model, q3_topics, q3_probs, q3_docs = fit_bertopic(
    df=df,
    column_index=10
)

2026-04-06 16:47:24,439 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic for prompt: What hypotheses might the experimenter have been testing


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13981.01it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:01<00:00, 10.21it/s]
2026-04-06 16:47:27,694 - BERTopic - Embedding - Completed ✓
2026-04-06 16:47:27,694 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 16:47:28,242 - BERTopic - Dimensionality - Completed ✓
2026-04-06 16:47:28,243 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 16:47:28,282 - BERTopic - Cluster - Completed ✓
2026-04-06 16:47:28,284 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 51/51 [00:37<00:00,  1.37it/s]
2026-04-06 16:48:05,746 - BERTop

In [25]:
q3_topic_model.visualize_topics()

In [26]:
q3_hierarchical_topics = q3_topic_model.hierarchical_topics(q3_docs)
q3_topic_model.visualize_hierarchy(hierarchical_topics=q3_hierarchical_topics)

100%|██████████| 49/49 [00:38<00:00,  1.27it/s]


#### Q4

**Prompt:**
> What sources of error are involved in this experiment?

In [27]:
# Q4
q4_topic_model, q4_topics, q4_probs, q4_docs = fit_bertopic(
    df=df,
    column_index=11
)

2026-04-06 16:48:44,866 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic for prompt: What sources of error are involved in this experiment


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12423.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:01<00:00, 10.96it/s]
2026-04-06 16:48:48,099 - BERTopic - Embedding - Completed ✓
2026-04-06 16:48:48,100 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 16:48:48,647 - BERTopic - Dimensionality - Completed ✓
2026-04-06 16:48:48,648 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 16:48:48,680 - BERTopic - Cluster - Completed ✓
2026-04-06 16:48:48,682 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 42/42 [00:30<00:00,  1.39it/s]
2026-04-06 16:49:19,002 - BERTop

In [28]:
q4_topic_model.visualize_topics()

In [29]:
q4_hierarchical_topics = q4_topic_model.hierarchical_topics(q4_docs)
q4_topic_model.visualize_hierarchy(hierarchical_topics=q4_hierarchical_topics)

100%|██████████| 40/40 [00:29<00:00,  1.35it/s]


#### Q5

**Prompt:**
> What variables were examined? For each variable, identify whether it was quantitative or categorical.

In [30]:
# Q5
q5_topic_model, q5_topics, q5_probs, q5_docs = fit_bertopic(
    df=df,
    column_index=12
)

2026-04-06 16:49:49,034 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic for prompt: What variables were examined For each variable identify whether it was quantitative or categorical


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11717.84it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:01<00:00, 10.67it/s]
2026-04-06 16:49:52,235 - BERTopic - Embedding - Completed ✓
2026-04-06 16:49:52,235 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 16:49:52,793 - BERTopic - Dimensionality - Completed ✓
2026-04-06 16:49:52,793 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 16:49:52,828 - BERTopic - Cluster - Completed ✓
2026-04-06 16:49:52,830 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 42/42 [00:31<00:00,  1.35it/s]
2026-04-06 16:50:24,174 - BERTop

In [31]:
q5_topic_model.visualize_topics()

In [32]:
q5_hierarchical_topics = q5_topic_model.hierarchical_topics(q5_docs)
q5_topic_model.visualize_hierarchy(hierarchical_topics=q5_hierarchical_topics)

100%|██████████| 40/40 [00:29<00:00,  1.36it/s]


#### Q6

**Prompt:**
> What elements of experimental design, such as randomization or the use of a control group, do you think were present in the experiment? Why?

In [33]:
# Q6
q6_topic_model, q6_topics, q6_probs, q6_docs = fit_bertopic(
    df=df,
    column_index=9
)

2026-04-06 16:50:54,365 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic for prompt: What elements of experimental design such as randomization or the use of a control group do you think were present in the experiment Why


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13545.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:01<00:00,  8.65it/s]
2026-04-06 16:50:57,939 - BERTopic - Embedding - Completed ✓
2026-04-06 16:50:57,939 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 16:50:58,499 - BERTopic - Dimensionality - Completed ✓
2026-04-06 16:50:58,500 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 16:50:58,527 - BERTopic - Cluster - Completed ✓
2026-04-06 16:50:58,528 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 35/35 [00:27<00:00,  1.27it/s]
2026-04-06 16:51:26,273 - BERTop

In [34]:
q6_topic_model.visualize_topics()

In [35]:
q6_hierarchical_topics = q6_topic_model.hierarchical_topics(q6_docs)
q6_topic_model.visualize_hierarchy(hierarchical_topics=q6_hierarchical_topics)

100%|██████████| 33/33 [00:27<00:00,  1.22it/s]


### Abstract Reflection

#### Q7

**Prompt:**
> What components of the experiment are clearer now than they were as a participant? What questions do you still have for the experimenter? Write 3-5 sentences reflecting on the abstract.

In [36]:
# Q7
q7_topic_model, q7_topics, q7_probs, q7_docs = fit_bertopic(
    df=df,
    column_index=4
)

Running BERTopic for prompt: What components of the experiment are clearer now than they were as a participant What questions do you still have for the experimenter Write 3 5 sentences reflecting on the abstract


2026-04-06 16:51:53,856 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12803.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 16/16 [00:03<00:00,  4.34it/s]
2026-04-06 16:51:59,187 - BERTopic - Embedding - Completed ✓
2026-04-06 16:51:59,187 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 16:51:59,722 - BERTopic - Dimensionality - Completed ✓
2026-04-06 16:51:59,723 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 16:51:59,752 - BERTopic - Cluster - Completed ✓
2026-04-06 16:51:59,754 - BERTopic - Representation - Fine-tuning topics using representation m

In [37]:
q7_topic_model.visualize_topics()

In [38]:
q7_hierarchical_topics = q7_topic_model.hierarchical_topics(q7_docs)
q7_topic_model.visualize_hierarchy(hierarchical_topics=q7_hierarchical_topics)

100%|██████████| 38/38 [00:48<00:00,  1.26s/it]


### Presentation Reflection

#### Q8

**Prompt:**
> How did the information you gained from the components of this project (participation, post-study reflection, extended abstract, presentation) differ?

In [39]:
# Q8
q8_topic_model, q8_topics, q8_probs, q8_docs = fit_bertopic(
    df=df,
    column_index=14
)

Running BERTopic for prompt: How did the information you gained from the components of this project participation post study reflection extended abstract presentation differ


2026-04-06 16:53:32,452 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13929.17it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 13/13 [00:02<00:00,  5.93it/s]
2026-04-06 16:53:36,264 - BERTopic - Embedding - Completed ✓
2026-04-06 16:53:36,264 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 16:53:36,686 - BERTopic - Dimensionality - Completed ✓
2026-04-06 16:53:36,687 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 16:53:36,704 - BERTopic - Cluster - Completed ✓
2026-04-06 16:53:36,706 - BERTopic - Representation - Fine-tuning topics using representation m

In [40]:
q8_topic_model.visualize_topics()

In [41]:
q8_hierarchical_topics = q8_topic_model.hierarchical_topics(q8_docs)
q8_topic_model.visualize_hierarchy(hierarchical_topics=q8_hierarchical_topics)

100%|██████████| 24/24 [00:21<00:00,  1.12it/s]


#### Q9

**Prompt:**
> What components were emphasized in the presentation that weren’t emphasized in the abstract? Why do you think that is?

In [42]:
# Q9
q9_topic_model, q9_topics, q9_probs, q9_docs = fit_bertopic(
    df=df,
    column_index=16
)

2026-04-06 16:54:22,075 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic for prompt: What components were emphasized in the presentation that weren t emphasized in the abstract Why do you think that is


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12294.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 13/13 [00:01<00:00,  7.64it/s]
2026-04-06 16:54:25,349 - BERTopic - Embedding - Completed ✓
2026-04-06 16:54:25,349 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 16:54:25,752 - BERTopic - Dimensionality - Completed ✓
2026-04-06 16:54:25,753 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 16:54:25,773 - BERTopic - Cluster - Completed ✓
2026-04-06 16:54:25,776 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 30/30 [00:25<00:00,  1.18it/s]
2026-04-06 16:54:51,385 - BERTop

In [43]:
q9_topic_model.visualize_topics()

In [44]:
q9_hierarchical_topics = q9_topic_model.hierarchical_topics(q9_docs)
q9_topic_model.visualize_hierarchy(hierarchical_topics=q9_hierarchical_topics)

100%|██████████| 28/28 [00:23<00:00,  1.17it/s]


#### Q10

**Prompt:**
> What critiques do you have of this study and its design? What would have made the study better?

In [45]:
# Q10
q10_topic_model, q10_topics, q10_probs, q10_docs = fit_bertopic(
    df=df,
    column_index=17
)

2026-04-06 16:55:15,792 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic for prompt: What critiques do you have of this study and its design What would have made the study better


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12200.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 13/13 [00:01<00:00,  7.69it/s]
2026-04-06 16:55:19,113 - BERTopic - Embedding - Completed ✓
2026-04-06 16:55:19,113 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 16:55:19,535 - BERTopic - Dimensionality - Completed ✓
2026-04-06 16:55:19,536 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 16:55:19,553 - BERTopic - Cluster - Completed ✓
2026-04-06 16:55:19,555 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 25/25 [00:20<00:00,  1.21it/s]
2026-04-06 16:55:40,404 - BERTop

In [46]:
q10_topic_model.visualize_topics()

In [47]:
q10_hierarchical_topics = q10_topic_model.hierarchical_topics(q10_docs)
q10_topic_model.visualize_hierarchy(hierarchical_topics=q10_hierarchical_topics)

100%|██████████| 23/23 [00:17<00:00,  1.30it/s]


#### Q11

**Prompt:**
> If you had to hear about this study using only the extended abstract or only the presentation, which one would you prefer? Which one would be better for determining whether the experiment was well designed?

In [48]:
# Q11
q11_topic_model, q11_topics, q11_probs, q11_docs = fit_bertopic(
    df=df,
    column_index=15
)

2026-04-06 16:55:58,553 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic for prompt: If you had to hear about this study using only the extended abstract or only the presentation which one would you prefer Which one would be better for determining whether the experiment was well designed


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11455.59it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 13/13 [00:01<00:00,  7.02it/s]
2026-04-06 16:56:02,087 - BERTopic - Embedding - Completed ✓
2026-04-06 16:56:02,088 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 16:56:02,503 - BERTopic - Dimensionality - Completed ✓
2026-04-06 16:56:02,503 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 16:56:02,521 - BERTopic - Cluster - Completed ✓
2026-04-06 16:56:02,523 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 25/25 [00:19<00:00,  1.27it/s]
2026-04-06 16:56:22,437 - BERTop

In [49]:
q11_topic_model.visualize_topics()

In [50]:
q11_hierarchical_topics = q11_topic_model.hierarchical_topics(q11_docs)
q11_topic_model.visualize_hierarchy(hierarchical_topics=q11_hierarchical_topics)

100%|██████████| 23/23 [00:16<00:00,  1.42it/s]
